# SCAG concept explanation: results explorer

Inspect saved evidence without retraining, reselecting neurons or running new ablations. Select the **cinema** Python kernel, then **Run All**. Edit `RUN_DIR` to inspect another explanation.

The analysis separates semantic association, coactivation and intervention sensitivity. These do not independently establish anatomical causality or explanation faithfulness. Previously inspected test patients remain exploratory.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display, Image, Markdown

RELATIVE_RUN = Path("scag_outputs/research/acdc/anatomy/20261109T154512_822872Z/resnet18/encoder.layer4/concept_explanations/MYO_20261111T101436_121849Z")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_concept_explanation.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the SCAG repository, or set ROOT explicitly.")
RUN_DIR = ROOT / RELATIVE_RUN
provenance = json.loads((RUN_DIR / "provenance.json").read_text())
concept = provenance["arguments"]["concept"]
gamma, tau = provenance["gamma"], provenance["tau"]
nodes = pd.read_csv(RUN_DIR / "nodes.csv")
edges = pd.read_csv(RUN_DIR / "edges.csv")
summary = pd.read_csv(RUN_DIR / "intervention_summary.csv")
patients = pd.read_csv(RUN_DIR / "intervention_patients.csv", dtype={"pid": str})
examples = pd.read_csv(RUN_DIR / "supporting_examples.csv", dtype={"pid": str})
with np.load(RUN_DIR / "overlay_arrays.npz") as data:
    overlays = dict(data)
assert nodes.neuron.is_unique
selected = set(nodes.neuron)
exact = patients[patients.strategy == "displayed_subgraph"]
assert exact.pid.is_unique and all(set(json.loads(v)) == selected for v in exact.neurons)
assert np.isfinite(patients[["kl", "flip", "margin_drop"]]).all().all()
assert not patients.duplicated(["pid", "strategy", "repeat"]).any()
print(f"{concept}: {len(nodes)} channels, {len(edges)} edges, {exact.pid.nunique()} evaluation patients")
print(RUN_DIR)


## 1. Open the explanation image

The PNG is embedded below. Open `explanation.png` or `explanation.pdf` in the run directory for full-resolution viewing. Graph labels were assigned on training data; they are not automatically validated test labels.

In [ ]:
display(Image(filename=str(RUN_DIR / "explanation.png"), width=1200))

## 2. Do the queried concept associations survive evaluation?

The horizontal threshold is frozen from the original protocol. Report raw scores too: crossing it is a descriptive check, not a significance test. A neuron can overlap MYO without being selective for MYO over other concepts.

In [ ]:
seeds = nodes[nodes.is_seed].copy()
concepts = [column.removeprefix("train_") for column in nodes if column.startswith("train_")]
other = [f"eval_{name}" for name in concepts if name != concept]
seeds["eval_selectivity"] = seeds[f"eval_{concept}"] - seeds[other].max(axis=1) if other else np.nan
seeds["retains_threshold"] = seeds[f"eval_{concept}"] >= gamma
display(seeds[["neuron", "community", "label", f"train_{concept}", f"eval_{concept}", "eval_selectivity", "retains_threshold"]])
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(seeds))
ax.bar(x - .18, seeds[f"train_{concept}"], .36, label="Train")
ax.bar(x + .18, seeds[f"eval_{concept}"], .36, label="Evaluation")
ax.axhline(gamma, color="black", linestyle="--", label=f"Frozen threshold {gamma}")
ax.set(xticks=x, xticklabels=[f"n{i}" for i in seeds.neuron], ylabel="Aggregate spatial IoU", title=f"{concept} seed associations")
ax.legend()
plt.show()
display(nodes)


## 3. Inspect the MRI images and exact masks

These arrays contain only the two channels and one seeded patient exported by the script—not every selected channel or every patient. Both phase images share the extracted activation mask. The target mask follows the run's concept definition (for anatomy, the union across ED/ES).

**If active pixels = 0, no cyan contour is expected.** This is not a rendering failure. It does not imply the neuron never activates on other patients.

In [ ]:
fig, axes = plt.subplots(len(overlays["neurons"]), 4, figsize=(13, 3.6 * len(overlays["neurons"])), squeeze=False)
overlap_rows = []
for row, neuron in enumerate(overlays["neurons"]):
    active = overlays["active"][row].astype(bool)
    target = overlays["target"][row].astype(bool)
    intersection, union = (active & target).sum(), (active | target).sum()
    for phase in (0, 1):
        axes[row, phase].imshow(overlays["images"][phase], cmap="gray")
        if active.any() and not active.all():
            axes[row, phase].contour(active, levels=[.5], colors="cyan", linewidths=.8)
        axes[row, phase].set_title(f"n{neuron}: {['ED','ES'][phase]}")
    axes[row, 2].imshow(target, cmap="gray", vmin=0, vmax=1)
    axes[row, 2].set_title("Concept mask")
    axes[row, 3].imshow(active, cmap="gray", vmin=0, vmax=1)
    axes[row, 3].set_title(f"Activation mask: {active.sum()} pixels")
    overlap_rows.append(dict(neuron=int(neuron), patient=str(overlays["pid"].item()), active_pixels=int(active.sum()), target_pixels=int(target.sum()), intersection=int(intersection), iou=float(intersection / union) if union else 0.0))
for ax in axes.flat:
    ax.axis("off")
fig.tight_layout()
plt.show()
display(pd.DataFrame(overlap_rows))


## 4. Trace graph connections

Edges are undirected correlations, not routes of causal transmission. All displayed edges were selected using training data. Passing the evaluation threshold does not establish statistical significance; the threshold is relatively permissive for some of these edges.

In [ ]:
display(edges.sort_values("train_r", ascending=False))
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(edges.train_r, edges.eval_r)
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.axhline(tau, color="orange", linestyle=":")
ax.axvline(tau, color="orange", linestyle=":")
ax.set(xlabel="Training correlation", ylabel="Evaluation correlation", title="Same selected edges, independent cohort", xlim=(0, 1), ylim=(0, 1))
plt.show()
# Change this identifier to inspect a seed or neighbour.
NEURON = int(seeds.iloc[0].neuron)
display(nodes[nodes.neuron == NEURON])
display(edges[(edges.source == NEURON) | (edges.target == NEURON)])
display(examples[(examples.neuron == NEURON) & (examples.concept == concept)].sort_values("rank"))
print("Supporting examples are overlap-selected; they are not a representative population sample.")


## 5. Intervention effects

`displayed_subgraph`, `semantic`, `functional` and each random draw remove the same number of channels. `query_seeds` removes fewer and is not a budget-matched comparator. KL is measured in nats, flip is a fraction, and signed margin decrease uses the original predicted class and original runner-up. Higher KL is not automatically a better explanation.

A zero observed flip rate and zero bootstrap interval do not prove a zero population flip probability.

In [ ]:
display(patients.groupby("strategy").agg(patients=("pid", "nunique"), min_channels=("k", "min"), max_channels=("k", "max"), selections=("repeat", "nunique")))
display(summary)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["kl", "flip", "margin_drop"]):
    data = summary[summary.metric == metric].reset_index(drop=True)
    ax.scatter(data["mean"], range(len(data)), color="teal")
    ax.hlines(range(len(data)), data.ci_low, data.ci_high, color="teal")
    ax.axvline(0, color="gray", linewidth=.6)
    ax.set(yticks=range(len(data)), yticklabels=data.strategy, xlabel=metric, title="Pointwise patient-bootstrap intervals")
fig.tight_layout()
plt.show()


## 6. Paired comparisons: displayed subgraph minus controls

Do not infer differences from overlapping marginal error bars. Compare strategies within each patient. Random draws are first averaged within patient. All comparisons below use identical patient bootstrap draws; intervals are exploratory, pointwise and not corrected for multiple comparisons. They condition on the trained model, fixed selections and the sampled random controls. They do not include training-seed or selection uncertainty.

In [ ]:
REPEATS, SEED = 5000, 42
metrics = ["kl", "flip", "margin_drop"]
means = patients.groupby(["pid", "strategy"])[metrics].mean()
ids = sorted(patients.pid.unique())
rng = np.random.default_rng(SEED)
draws = rng.integers(len(ids), size=(REPEATS, len(ids)))
contrast_rows = []
for metric in metrics:
    table = means[metric].unstack().reindex(ids)
    assert not table.isna().any().any()
    for control in ["semantic", "functional", "random"]:
        delta = (table.displayed_subgraph - table[control]).to_numpy()
        lo, hi = np.quantile(delta[draws].mean(axis=1), [.025, .975])
        contrast_rows.append(dict(metric=metric, control=control, difference=delta.mean(), ci_low=lo, ci_high=hi, excludes_zero=bool(lo > 0 or hi < 0)))
contrasts = pd.DataFrame(contrast_rows)
display(contrasts)
fig, ax = plt.subplots(figsize=(7, 3))
v = contrasts[contrasts.metric == "kl"].reset_index(drop=True)
ax.scatter(v.difference, range(len(v)), color="teal")
ax.hlines(range(len(v)), v.ci_low, v.ci_high, color="teal")
ax.axvline(0, color="black", linestyle="--")
ax.set(yticks=range(len(v)), yticklabels=v.control, xlabel="Paired KL difference: displayed minus control", title="Positive means greater disruption, not greater faithfulness")
plt.show()


## 7. Patient heterogeneity and selection overlap

Group averages can conceal opposite signed effects. Selection overlap also matters: nearly identical channel sets are not strong tests of complementary information.

In [ ]:
table = means["margin_drop"].unstack().reindex(ids)
fig, ax = plt.subplots(figsize=(9, 4))
for strategy in ["displayed_subgraph", "semantic", "functional"]:
    ax.plot(np.sort(table[strategy]), label=strategy)
ax.axhline(0, color="gray", linestyle="--")
ax.set(xlabel="Patient rank within each strategy (not paired across curves)", ylabel="Signed margin decrease")
ax.legend()
plt.show()
overlap = []
for strategy in ["semantic", "functional"]:
    channels = set(json.loads(patients.loc[patients.strategy == strategy, "neurons"].iloc[0]))
    overlap.append(dict(control=strategy, shared_channels=len(selected & channels), jaccard=len(selected & channels) / len(selected | channels)))
display(pd.DataFrame(overlap))


## 8. Interpretation and next step

For this MYO run, the functional neighbourhood persists but the anatomical labels do not meet the fixed test threshold. The ablation shifts probability distributions without changing any predicted label for the displayed set. Its KL effect is larger than random, but the paired intervals do not establish a clear advantage over semantic-only or functional-only selection.

The pipeline is producing inspectable explanations; it has **not yet demonstrated a reliable MYO-specific explanation or added value of joint selection**. Investigate semantic calibration, activation coverage and patient-level overlap before expanding to cross-layer tracing. Do not lower the test threshold just to make the labels survive.

The saved high-overlap examples and the representative zero-activation image answer different questions; inspect both. The next audit should explain the train/evaluation association decrease, not hide it with selected examples.

### Your notes

- What looks convincing?
- Which concept labels need more evidence?
- What should be discussed with the supervisor?


<!-- multi-concept-analysis -->
# LV and RV explanations

The following sections inspect the completed LV and RV runs with the same checks as MYO. These are concept-query neighbourhoods, not whole-community ablations. All use nine displayed channels and the same 50 patients.

# LV: evidence and intervention analysis

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display, Image, Markdown

RELATIVE_RUN = Path("scag_outputs/research/acdc/anatomy/20261109T154512_822872Z/resnet18/encoder.layer4/concept_explanations/LV_20261111T103335_808315Z")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_concept_explanation.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the SCAG repository, or set ROOT explicitly.")
RUN_DIR = ROOT / RELATIVE_RUN
provenance = json.loads((RUN_DIR / "provenance.json").read_text())
concept = provenance["arguments"]["concept"]
gamma, tau = provenance["gamma"], provenance["tau"]
nodes = pd.read_csv(RUN_DIR / "nodes.csv")
edges = pd.read_csv(RUN_DIR / "edges.csv")
summary = pd.read_csv(RUN_DIR / "intervention_summary.csv")
patients = pd.read_csv(RUN_DIR / "intervention_patients.csv", dtype={"pid": str})
examples = pd.read_csv(RUN_DIR / "supporting_examples.csv", dtype={"pid": str})
with np.load(RUN_DIR / "overlay_arrays.npz") as data:
    overlays = dict(data)
assert nodes.neuron.is_unique
selected = set(nodes.neuron)
exact = patients[patients.strategy == "displayed_subgraph"]
assert exact.pid.is_unique and all(set(json.loads(v)) == selected for v in exact.neurons)
assert np.isfinite(patients[["kl", "flip", "margin_drop"]]).all().all()
assert not patients.duplicated(["pid", "strategy", "repeat"]).any()
print(f"{concept}: {len(nodes)} channels, {len(edges)} edges, {exact.pid.nunique()} evaluation patients")
print(RUN_DIR)


## 1. Open the explanation image

The PNG is embedded below. Open `explanation.png` or `explanation.pdf` in the run directory for full-resolution viewing. Graph labels were assigned on training data; they are not automatically validated test labels.

In [ ]:
display(Image(filename=str(RUN_DIR / "explanation.png"), width=1200))

## 2. Do the queried concept associations survive evaluation?

The horizontal threshold is frozen from the original protocol. Report raw scores too: crossing it is a descriptive check, not a significance test. A neuron can overlap MYO without being selective for MYO over other concepts.

In [ ]:
seeds = nodes[nodes.is_seed].copy()
concepts = [column.removeprefix("train_") for column in nodes if column.startswith("train_")]
other = [f"eval_{name}" for name in concepts if name != concept]
seeds["eval_selectivity"] = seeds[f"eval_{concept}"] - seeds[other].max(axis=1) if other else np.nan
seeds["retains_threshold"] = seeds[f"eval_{concept}"] >= gamma
display(seeds[["neuron", "community", "label", f"train_{concept}", f"eval_{concept}", "eval_selectivity", "retains_threshold"]])
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(seeds))
ax.bar(x - .18, seeds[f"train_{concept}"], .36, label="Train")
ax.bar(x + .18, seeds[f"eval_{concept}"], .36, label="Evaluation")
ax.axhline(gamma, color="black", linestyle="--", label=f"Frozen threshold {gamma}")
ax.set(xticks=x, xticklabels=[f"n{i}" for i in seeds.neuron], ylabel="Aggregate spatial IoU", title=f"{concept} seed associations")
ax.legend()
plt.show()
display(nodes)


## 3. Inspect the MRI images and exact masks

These arrays contain only the two channels and one seeded patient exported by the script—not every selected channel or every patient. Both phase images share the extracted activation mask. The target mask follows the run's concept definition (for anatomy, the union across ED/ES).

**If active pixels = 0, no cyan contour is expected.** This is not a rendering failure. It does not imply the neuron never activates on other patients.

In [ ]:
fig, axes = plt.subplots(len(overlays["neurons"]), 4, figsize=(13, 3.6 * len(overlays["neurons"])), squeeze=False)
overlap_rows = []
for row, neuron in enumerate(overlays["neurons"]):
    active = overlays["active"][row].astype(bool)
    target = overlays["target"][row].astype(bool)
    intersection, union = (active & target).sum(), (active | target).sum()
    for phase in (0, 1):
        axes[row, phase].imshow(overlays["images"][phase], cmap="gray")
        if active.any() and not active.all():
            axes[row, phase].contour(active, levels=[.5], colors="cyan", linewidths=.8)
        axes[row, phase].set_title(f"n{neuron}: {['ED','ES'][phase]}")
    axes[row, 2].imshow(target, cmap="gray", vmin=0, vmax=1)
    axes[row, 2].set_title("Concept mask")
    axes[row, 3].imshow(active, cmap="gray", vmin=0, vmax=1)
    axes[row, 3].set_title(f"Activation mask: {active.sum()} pixels")
    overlap_rows.append(dict(neuron=int(neuron), patient=str(overlays["pid"].item()), active_pixels=int(active.sum()), target_pixels=int(target.sum()), intersection=int(intersection), iou=float(intersection / union) if union else 0.0))
for ax in axes.flat:
    ax.axis("off")
fig.tight_layout()
plt.show()
display(pd.DataFrame(overlap_rows))


## 4. Trace graph connections

Edges are undirected correlations, not routes of causal transmission. All displayed edges were selected using training data. Passing the evaluation threshold does not establish statistical significance; the threshold is relatively permissive for some of these edges.

In [ ]:
display(edges.sort_values("train_r", ascending=False))
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(edges.train_r, edges.eval_r)
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.axhline(tau, color="orange", linestyle=":")
ax.axvline(tau, color="orange", linestyle=":")
ax.set(xlabel="Training correlation", ylabel="Evaluation correlation", title="Same selected edges, independent cohort", xlim=(0, 1), ylim=(0, 1))
plt.show()
# Change this identifier to inspect a seed or neighbour.
NEURON = int(seeds.iloc[0].neuron)
display(nodes[nodes.neuron == NEURON])
display(edges[(edges.source == NEURON) | (edges.target == NEURON)])
display(examples[(examples.neuron == NEURON) & (examples.concept == concept)].sort_values("rank"))
print("Supporting examples are overlap-selected; they are not a representative population sample.")


## 5. Intervention effects

`displayed_subgraph`, `semantic`, `functional` and each random draw remove the same number of channels. `query_seeds` removes fewer and is not a budget-matched comparator. KL is measured in nats, flip is a fraction, and signed margin decrease uses the original predicted class and original runner-up. Higher KL is not automatically a better explanation.

A zero observed flip rate and zero bootstrap interval do not prove a zero population flip probability.

In [ ]:
display(patients.groupby("strategy").agg(patients=("pid", "nunique"), min_channels=("k", "min"), max_channels=("k", "max"), selections=("repeat", "nunique")))
display(summary)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["kl", "flip", "margin_drop"]):
    data = summary[summary.metric == metric].reset_index(drop=True)
    ax.scatter(data["mean"], range(len(data)), color="teal")
    ax.hlines(range(len(data)), data.ci_low, data.ci_high, color="teal")
    ax.axvline(0, color="gray", linewidth=.6)
    ax.set(yticks=range(len(data)), yticklabels=data.strategy, xlabel=metric, title="Pointwise patient-bootstrap intervals")
fig.tight_layout()
plt.show()


## 6. Paired comparisons: displayed subgraph minus controls

Do not infer differences from overlapping marginal error bars. Compare strategies within each patient. Random draws are first averaged within patient. All comparisons below use identical patient bootstrap draws; intervals are exploratory, pointwise and not corrected for multiple comparisons. They condition on the trained model, fixed selections and the sampled random controls. They do not include training-seed or selection uncertainty.

In [ ]:
REPEATS, SEED = 5000, 42
metrics = ["kl", "flip", "margin_drop"]
means = patients.groupby(["pid", "strategy"])[metrics].mean()
ids = sorted(patients.pid.unique())
rng = np.random.default_rng(SEED)
draws = rng.integers(len(ids), size=(REPEATS, len(ids)))
contrast_rows = []
for metric in metrics:
    table = means[metric].unstack().reindex(ids)
    assert not table.isna().any().any()
    for control in ["semantic", "functional", "random"]:
        delta = (table.displayed_subgraph - table[control]).to_numpy()
        lo, hi = np.quantile(delta[draws].mean(axis=1), [.025, .975])
        contrast_rows.append(dict(metric=metric, control=control, difference=delta.mean(), ci_low=lo, ci_high=hi, excludes_zero=bool(lo > 0 or hi < 0)))
contrasts = pd.DataFrame(contrast_rows)
display(contrasts)
fig, ax = plt.subplots(figsize=(7, 3))
v = contrasts[contrasts.metric == "kl"].reset_index(drop=True)
ax.scatter(v.difference, range(len(v)), color="teal")
ax.hlines(range(len(v)), v.ci_low, v.ci_high, color="teal")
ax.axvline(0, color="black", linestyle="--")
ax.set(yticks=range(len(v)), yticklabels=v.control, xlabel="Paired KL difference: displayed minus control", title="Positive means greater disruption, not greater faithfulness")
plt.show()


## 7. Patient heterogeneity and selection overlap

Group averages can conceal opposite signed effects. Selection overlap also matters: nearly identical channel sets are not strong tests of complementary information.

In [ ]:
table = means["margin_drop"].unstack().reindex(ids)
fig, ax = plt.subplots(figsize=(9, 4))
for strategy in ["displayed_subgraph", "semantic", "functional"]:
    ax.plot(np.sort(table[strategy]), label=strategy)
ax.axhline(0, color="gray", linestyle="--")
ax.set(xlabel="Patient rank within each strategy (not paired across curves)", ylabel="Signed margin decrease")
ax.legend()
plt.show()
overlap = []
for strategy in ["semantic", "functional"]:
    channels = set(json.loads(patients.loc[patients.strategy == strategy, "neurons"].iloc[0]))
    overlap.append(dict(control=strategy, shared_channels=len(selected & channels), jaccard=len(selected & channels) / len(selected | channels)))
display(pd.DataFrame(overlap))


# RV: evidence and intervention analysis

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display, Image, Markdown

RELATIVE_RUN = Path("scag_outputs/research/acdc/anatomy/20261109T154512_822872Z/resnet18/encoder.layer4/concept_explanations/RV_20261111T103638_162078Z")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_concept_explanation.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the SCAG repository, or set ROOT explicitly.")
RUN_DIR = ROOT / RELATIVE_RUN
provenance = json.loads((RUN_DIR / "provenance.json").read_text())
concept = provenance["arguments"]["concept"]
gamma, tau = provenance["gamma"], provenance["tau"]
nodes = pd.read_csv(RUN_DIR / "nodes.csv")
edges = pd.read_csv(RUN_DIR / "edges.csv")
summary = pd.read_csv(RUN_DIR / "intervention_summary.csv")
patients = pd.read_csv(RUN_DIR / "intervention_patients.csv", dtype={"pid": str})
examples = pd.read_csv(RUN_DIR / "supporting_examples.csv", dtype={"pid": str})
with np.load(RUN_DIR / "overlay_arrays.npz") as data:
    overlays = dict(data)
assert nodes.neuron.is_unique
selected = set(nodes.neuron)
exact = patients[patients.strategy == "displayed_subgraph"]
assert exact.pid.is_unique and all(set(json.loads(v)) == selected for v in exact.neurons)
assert np.isfinite(patients[["kl", "flip", "margin_drop"]]).all().all()
assert not patients.duplicated(["pid", "strategy", "repeat"]).any()
print(f"{concept}: {len(nodes)} channels, {len(edges)} edges, {exact.pid.nunique()} evaluation patients")
print(RUN_DIR)


## 1. Open the explanation image

The PNG is embedded below. Open `explanation.png` or `explanation.pdf` in the run directory for full-resolution viewing. Graph labels were assigned on training data; they are not automatically validated test labels.

In [ ]:
display(Image(filename=str(RUN_DIR / "explanation.png"), width=1200))

## 2. Do the queried concept associations survive evaluation?

The horizontal threshold is frozen from the original protocol. Report raw scores too: crossing it is a descriptive check, not a significance test. A neuron can overlap MYO without being selective for MYO over other concepts.

In [ ]:
seeds = nodes[nodes.is_seed].copy()
concepts = [column.removeprefix("train_") for column in nodes if column.startswith("train_")]
other = [f"eval_{name}" for name in concepts if name != concept]
seeds["eval_selectivity"] = seeds[f"eval_{concept}"] - seeds[other].max(axis=1) if other else np.nan
seeds["retains_threshold"] = seeds[f"eval_{concept}"] >= gamma
display(seeds[["neuron", "community", "label", f"train_{concept}", f"eval_{concept}", "eval_selectivity", "retains_threshold"]])
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(seeds))
ax.bar(x - .18, seeds[f"train_{concept}"], .36, label="Train")
ax.bar(x + .18, seeds[f"eval_{concept}"], .36, label="Evaluation")
ax.axhline(gamma, color="black", linestyle="--", label=f"Frozen threshold {gamma}")
ax.set(xticks=x, xticklabels=[f"n{i}" for i in seeds.neuron], ylabel="Aggregate spatial IoU", title=f"{concept} seed associations")
ax.legend()
plt.show()
display(nodes)


## 3. Inspect the MRI images and exact masks

These arrays contain only the two channels and one seeded patient exported by the script—not every selected channel or every patient. Both phase images share the extracted activation mask. The target mask follows the run's concept definition (for anatomy, the union across ED/ES).

**If active pixels = 0, no cyan contour is expected.** This is not a rendering failure. It does not imply the neuron never activates on other patients.

In [ ]:
fig, axes = plt.subplots(len(overlays["neurons"]), 4, figsize=(13, 3.6 * len(overlays["neurons"])), squeeze=False)
overlap_rows = []
for row, neuron in enumerate(overlays["neurons"]):
    active = overlays["active"][row].astype(bool)
    target = overlays["target"][row].astype(bool)
    intersection, union = (active & target).sum(), (active | target).sum()
    for phase in (0, 1):
        axes[row, phase].imshow(overlays["images"][phase], cmap="gray")
        if active.any() and not active.all():
            axes[row, phase].contour(active, levels=[.5], colors="cyan", linewidths=.8)
        axes[row, phase].set_title(f"n{neuron}: {['ED','ES'][phase]}")
    axes[row, 2].imshow(target, cmap="gray", vmin=0, vmax=1)
    axes[row, 2].set_title("Concept mask")
    axes[row, 3].imshow(active, cmap="gray", vmin=0, vmax=1)
    axes[row, 3].set_title(f"Activation mask: {active.sum()} pixels")
    overlap_rows.append(dict(neuron=int(neuron), patient=str(overlays["pid"].item()), active_pixels=int(active.sum()), target_pixels=int(target.sum()), intersection=int(intersection), iou=float(intersection / union) if union else 0.0))
for ax in axes.flat:
    ax.axis("off")
fig.tight_layout()
plt.show()
display(pd.DataFrame(overlap_rows))


## 4. Trace graph connections

Edges are undirected correlations, not routes of causal transmission. All displayed edges were selected using training data. Passing the evaluation threshold does not establish statistical significance; the threshold is relatively permissive for some of these edges.

In [ ]:
display(edges.sort_values("train_r", ascending=False))
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(edges.train_r, edges.eval_r)
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.axhline(tau, color="orange", linestyle=":")
ax.axvline(tau, color="orange", linestyle=":")
ax.set(xlabel="Training correlation", ylabel="Evaluation correlation", title="Same selected edges, independent cohort", xlim=(0, 1), ylim=(0, 1))
plt.show()
# Change this identifier to inspect a seed or neighbour.
NEURON = int(seeds.iloc[0].neuron)
display(nodes[nodes.neuron == NEURON])
display(edges[(edges.source == NEURON) | (edges.target == NEURON)])
display(examples[(examples.neuron == NEURON) & (examples.concept == concept)].sort_values("rank"))
print("Supporting examples are overlap-selected; they are not a representative population sample.")


## 5. Intervention effects

`displayed_subgraph`, `semantic`, `functional` and each random draw remove the same number of channels. `query_seeds` removes fewer and is not a budget-matched comparator. KL is measured in nats, flip is a fraction, and signed margin decrease uses the original predicted class and original runner-up. Higher KL is not automatically a better explanation.

A zero observed flip rate and zero bootstrap interval do not prove a zero population flip probability.

In [ ]:
display(patients.groupby("strategy").agg(patients=("pid", "nunique"), min_channels=("k", "min"), max_channels=("k", "max"), selections=("repeat", "nunique")))
display(summary)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["kl", "flip", "margin_drop"]):
    data = summary[summary.metric == metric].reset_index(drop=True)
    ax.scatter(data["mean"], range(len(data)), color="teal")
    ax.hlines(range(len(data)), data.ci_low, data.ci_high, color="teal")
    ax.axvline(0, color="gray", linewidth=.6)
    ax.set(yticks=range(len(data)), yticklabels=data.strategy, xlabel=metric, title="Pointwise patient-bootstrap intervals")
fig.tight_layout()
plt.show()


## 6. Paired comparisons: displayed subgraph minus controls

Do not infer differences from overlapping marginal error bars. Compare strategies within each patient. Random draws are first averaged within patient. All comparisons below use identical patient bootstrap draws; intervals are exploratory, pointwise and not corrected for multiple comparisons. They condition on the trained model, fixed selections and the sampled random controls. They do not include training-seed or selection uncertainty.

In [ ]:
REPEATS, SEED = 5000, 42
metrics = ["kl", "flip", "margin_drop"]
means = patients.groupby(["pid", "strategy"])[metrics].mean()
ids = sorted(patients.pid.unique())
rng = np.random.default_rng(SEED)
draws = rng.integers(len(ids), size=(REPEATS, len(ids)))
contrast_rows = []
for metric in metrics:
    table = means[metric].unstack().reindex(ids)
    assert not table.isna().any().any()
    for control in ["semantic", "functional", "random"]:
        delta = (table.displayed_subgraph - table[control]).to_numpy()
        lo, hi = np.quantile(delta[draws].mean(axis=1), [.025, .975])
        contrast_rows.append(dict(metric=metric, control=control, difference=delta.mean(), ci_low=lo, ci_high=hi, excludes_zero=bool(lo > 0 or hi < 0)))
contrasts = pd.DataFrame(contrast_rows)
display(contrasts)
fig, ax = plt.subplots(figsize=(7, 3))
v = contrasts[contrasts.metric == "kl"].reset_index(drop=True)
ax.scatter(v.difference, range(len(v)), color="teal")
ax.hlines(range(len(v)), v.ci_low, v.ci_high, color="teal")
ax.axvline(0, color="black", linestyle="--")
ax.set(yticks=range(len(v)), yticklabels=v.control, xlabel="Paired KL difference: displayed minus control", title="Positive means greater disruption, not greater faithfulness")
plt.show()


## 7. Patient heterogeneity and selection overlap

Group averages can conceal opposite signed effects. Selection overlap also matters: nearly identical channel sets are not strong tests of complementary information.

In [ ]:
table = means["margin_drop"].unstack().reindex(ids)
fig, ax = plt.subplots(figsize=(9, 4))
for strategy in ["displayed_subgraph", "semantic", "functional"]:
    ax.plot(np.sort(table[strategy]), label=strategy)
ax.axhline(0, color="gray", linestyle="--")
ax.set(xlabel="Patient rank within each strategy (not paired across curves)", ylabel="Signed margin decrease")
ax.legend()
plt.show()
overlap = []
for strategy in ["semantic", "functional"]:
    channels = set(json.loads(patients.loc[patients.strategy == strategy, "neurons"].iloc[0]))
    overlap.append(dict(control=strategy, shared_channels=len(selected & channels), jaccard=len(selected & channels) / len(selected | channels)))
display(pd.DataFrame(overlap))


# Comparison across LV, MYO and RV

Seed IoU and vocabulary-relative selectivity assess semantic association; edge retention assesses threshold persistence, not causal validity. The nine-channel interventions are budget-matched. Functional and random controls are shared across concepts, so their repeated values are not independent replications.

Paired bootstrap intervals below use 5,000 patient resamples, averaging random draws within patient first. They are exploratory pointwise 95% intervals without multiple-comparison correction. Positive margin drop means a decrease in the original predicted-class versus original runner-up margin; negative values mean an increase, not necessarily improved accuracy.

In [ ]:
RUN_NAMES = {"LV": "LV_20261111T103335_808315Z", "MYO": "MYO_20261111T101436_121849Z", "RV": "RV_20261111T103638_162078Z"}
base = RUN_DIR.parent
comparison_rows, seed_rows, contrast_rows, effect_rows = [], [], [], []
reference_ids, reference_hash = None, None
for name, folder in RUN_NAMES.items():
    directory = base / folder
    meta = json.loads((directory / "provenance.json").read_text())
    nd = pd.read_csv(directory / "nodes.csv")
    ed = pd.read_csv(directory / "edges.csv")
    pt = pd.read_csv(directory / "intervention_patients.csv", dtype={"pid": str})
    sm = pd.read_csv(directory / "intervention_summary.csv")
    ids = sorted(pt.pid.unique())
    if reference_ids is None:
        reference_ids, reference_hash = ids, meta["representation_sha256"]
    assert ids == reference_ids and meta["representation_sha256"] == reference_hash
    assert not pt.duplicated(["pid", "strategy", "repeat"]).any()
    assert np.isfinite(pt[["kl", "flip", "margin_drop"]]).all().all()
    exact = pt[pt.strategy == "displayed_subgraph"]
    assert exact.pid.is_unique and len(exact) == len(ids)
    assert all(set(json.loads(value)) == set(nd.neuron) for value in exact.neurons)
    assert pt[pt.strategy != "query_seeds"].k.eq(len(nd)).all()
    sd = nd[nd.is_seed].copy()
    others = [c for c in nd if c.startswith("eval_") and c != f"eval_{name}"]
    selectivity = sd[f"eval_{name}"] - sd[others].max(axis=1)
    comparison_rows.append(dict(concept=name, channels=len(nd), seeds=len(sd), retained_seeds=int(sd[f"eval_{name}"].ge(meta["gamma"]).sum()), edges=len(ed), retained_edges=int(ed.eval_r.ge(meta["tau"]).sum()), mean_seed_test_iou=sd[f"eval_{name}"].mean(), mean_seed_selectivity=selectivity.mean()))
    for index, row in sd.iterrows():
        seed_rows.append(dict(concept=name, neuron=int(row.neuron), community=int(row.community), train_iou=row[f"train_{name}"], test_iou=row[f"eval_{name}"], test_selectivity=selectivity.loc[index]))
    effect_rows.append(sm.assign(concept=name))
    means = pt.groupby(["pid", "strategy"])[["kl", "flip", "margin_drop"]].mean()
    draws = np.random.default_rng(42).integers(0, len(ids), (5000, len(ids)))
    for metric in ["kl", "flip", "margin_drop"]:
        wide = means[metric].unstack().reindex(ids)
        assert wide.notna().all().all()
        for control in ["semantic", "functional", "random"]:
            delta = (wide["displayed_subgraph"] - wide[control]).to_numpy()
            low, high = np.quantile(delta[draws].mean(axis=1), [.025, .975])
            contrast_rows.append(dict(concept=name, metric=metric, control=control, difference=delta.mean(), ci_low=low, ci_high=high))
comparison = pd.DataFrame(comparison_rows)
seed_comparison = pd.DataFrame(seed_rows)
all_effects = pd.concat(effect_rows, ignore_index=True)
all_contrasts = pd.DataFrame(contrast_rows)
display(comparison)
display(seed_comparison)
display(all_effects[all_effects.strategy == "displayed_subgraph"])
display(all_contrasts)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["kl", "flip", "margin_drop"]):
    selected = all_contrasts[all_contrasts.metric == metric].reset_index(drop=True)
    for i, row in selected.iterrows():
        ax.plot([row.ci_low, row.ci_high], [i, i], color="tab:blue")
        ax.scatter(row.difference, i, color="tab:blue")
    ax.set_yticks(range(len(selected)), selected.concept + " − " + selected.control)
    ax.axvline(0, color="gray", linestyle="--")
    ax.set_title(f"Displayed subgraph minus control: {metric}")
    ax.invert_yaxis()
fig.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(8, 4))
for name, group in seed_comparison.groupby("concept", sort=False):
    ax.scatter(group.train_iou, group.test_iou, label=name)
    for row in group.itertuples():
        ax.annotate(str(row.neuron), (row.train_iou, row.test_iou), fontsize=8)
ax.axhline(.015, color="gray", linestyle="--", label="Frozen association threshold")
ax.set(xlabel="Training seed IoU", ylabel="Test seed IoU", title="Semantic association retention")
ax.legend()
plt.show()


## Interpretation for these three saved runs

- **LV is the strongest semantic example:** all three seeds retain the fixed association threshold, although only 22/32 edges do. Ablation reduces the original class-pair margin on average. Inspect the paired intervals above before claiming superiority over a control.
- **RV has partial semantic retention:** one of three seeds retains the threshold; all 18 edges persist. Its negative mean margin drop means the intervention increases the original class-pair margin on average. This is sensitivity, not evidence that RV-associated channels support the original prediction or that accuracy improves.
- **MYO retains functional structure but weak semantic grounding:** zero of three seed labels and all 18 edges retain their respective thresholds.
- For a paper, show all three in a consistent comparison. LV can serve as a detailed illustrative case, explicitly selected after inspecting results; MYO and RV reveal limits. None is a demonstrated causal pathway, and a concept-query subgraph is not the same as a whole community.

### Comments and follow-up

Record supervisor comments here. Improve visual evidence using a documented example-selection rule, retaining low/zero-overlap examples and the frozen calibration. No new inference or ablations are performed by this notebook.
